# Tenacious-Bench v0.1: LoRA Training on Qwen2.5-0.5B-Instruct

**Target:** Mitigate Signal Over-Claiming (phrasing-tier calibration) via SFT + LoRA.  
**Hardware:** T4 GPU, 16 GB VRAM.  
**Expected wall-clock:** ~15 min training + ~2 min ablation.  

Run cells **sequentially**. Do not skip Step 4 (HF login) if you want the adapter pushed to the Hub.

## Step 1: Install Dependencies

In [ ]:
# Uninstall Colab pre-installed bitsandbytes — its triton.ops import breaks on
# Colab current triton. We use fp16 LoRA, so bnb is not needed.
!pip uninstall -y bitsandbytes 2>/dev/null

# Pinned dependencies — must match requirements.txt versions so SFTConfig/SFTTrainer
# signatures stay stable. Do NOT install unsloth: lora_train.py uses standard PEFT+TRL.
!pip install -q   "transformers==4.47.0"   "trl==0.12.0"   "peft==0.13.0"   "accelerate==1.2.0"   "datasets==3.2.0"   "huggingface_hub==0.26.5"   "python-dotenv==1.0.1"
print('Dependencies installed.')

## Step 1b: Verify environment
*(Sanity check — if anything prints UNEXPECTED, restart the runtime and re-run Step 1.)*

In [ ]:
# Sanity check — confirm the SFT API surface lora_train.py expects is present.
# If any line below prints UNEXPECTED, STOP and re-run Step 1.
import inspect, transformers, trl, peft, accelerate, datasets
from trl import SFTConfig, SFTTrainer
sft_params = set(inspect.signature(SFTConfig.__init__).parameters)
trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
print(f'transformers   {transformers.__version__}')
print(f'trl            {trl.__version__}')
print(f'peft           {peft.__version__}')
print(f'accelerate     {accelerate.__version__}')
print(f'datasets       {datasets.__version__}')
print(f'SFTConfig has  max_seq_length: {"max_seq_length" in sft_params}  max_length: {"max_length" in sft_params}')
print(f'SFTTrainer has tokenizer: {"tokenizer" in trainer_params}  processing_class: {"processing_class" in trainer_params}')
import torch
assert torch.cuda.is_available(), "GPU NOT VISIBLE — switch Colab runtime to T4 GPU before continuing."
print(f'CUDA OK: {torch.cuda.get_device_name(0)}')

## Step 2: Clone Repository

In [ ]:
!git clone https://github.com/78gk/Sales-Agent-Evaluation-Bench.git
%cd Sales-Agent-Evaluation-Bench
!git log --oneline -3
print('Repo cloned.')

## Step 3: Prepare SFT Data
*(LIMA-filtered paraphrase augmentation: 125 tasks × 21x = ~2,625 ChatML pairs)*

In [ ]:
!python training/prepare_sft_data.py \
    --tasks-dir tenacious_bench_v0.1/train \
    --output training/qwen_pairs.jsonl \
    --shuffle
!cat training/pair_count.txt

## Step 4: Login to Hugging Face
*(Paste your WRITE-scoped token from hf.co/settings/tokens when prompted)*

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 5: Run LoRA Training
*(Rank 16, Alpha 32, q_proj+v_proj, 3 epochs, cosine LR. Estimated time: ~15 min.  
Kill criterion: if wall-clock > 30 min the script exits and logs the warning — see CLAUDE.md §kill-criterion.)*

In [ ]:
!python training/lora_train.py \
    --pairs training/qwen_pairs.jsonl \
    --model qwen2.5-0.5b-instruct \
    --output training/checkpoint \
    --hub-repo kirutew17654321/tenacious-bench-qwen-lora

## Step 6: Upload Sealed Held-Out Tasks
*(The held-out split is gitignored. Upload `held_out.zip` — a zip of your local `tenacious_bench_v0.1/held_out/` folder.  
If you skip this step, the ablation will run with `--dry-run` mock outputs instead of the real adapter.)*

In [ ]:
import os
from google.colab import files

# Option A: Upload held_out.zip from your machine
print('Upload held_out.zip (zip of tenacious_bench_v0.1/held_out/ on your local machine).')
print('If you skip the upload, the ablation will use --dry-run mock outputs.')

try:
    uploaded = files.upload()  # prompts file picker
    if 'held_out.zip' in uploaded:
        !unzip -q held_out.zip -d tenacious_bench_v0.1/
        count = len(os.listdir('tenacious_bench_v0.1/held_out'))
        print(f'Uploaded {count} held-out tasks.')
        USE_DRY_RUN = False
    else:
        print('held_out.zip not found — ablation will use --dry-run.')
        USE_DRY_RUN = True
except Exception:
    print('Upload skipped — ablation will use --dry-run.')
    USE_DRY_RUN = True

## Step 7: Run Delta A / Delta B Ablation
*(Scores the trained adapter on held_out. Primary result: Delta A vs Week 10 baseline, p-value via paired bootstrap n=1000.)*

In [ ]:
dry_run_flag = '--dry-run' if USE_DRY_RUN else ''

!python training/run_ablation.py \
    --held-out tenacious_bench_v0.1/held_out \
    --adapter training/checkpoint \
    --model qwen2.5-0.5b-instruct \
    --output ablations/ablation_results.json \
    {dry_run_flag}

if USE_DRY_RUN:
    print('NOTE: Results above are from mock outputs. Upload held_out.zip and re-run for real Delta A.')

## Step 8: Download Results
*(Zips training loss log + ablation results + updated evidence_graph. Download and commit to the repo.)*

In [ ]:
from google.colab import files as colab_files

!zip -r training_results.zip \
    training/checkpoint/loss_log.json \
    ablations/ablation_results.json \
    evidence_graph.json

colab_files.download('training_results.zip')
print('Download started. Unzip and commit to your local repo.')